# Estimación Cuántica de Fase (QPE)

**Objetivo:** estimar la fase φ de un operador unitario U tal que U|ψ⟩ = e^(2πiφ)|ψ⟩.

QPE es el bloque fundamental de Shor, HHL y la simulación cuántica de Hamiltoniano.
La precisión depende del número de qubits ancilla: error < 2^(-n_ancilla).

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Ejemplo: U = T-gate, φ = 1/8 (T|1⟩ = e^(iπ/4)|1⟩ = e^(2πi·1/8)|1⟩)
# Usamos 3 ancillas → estimaremos φ con 3 bits de precisión

n_ancilla = 3
n_state = 1
qc = QuantumCircuit(n_ancilla + n_state, n_ancilla)

# Estado inicial del registro ancilla: superposición uniforme
for i in range(n_ancilla):
    qc.h(i)

# Estado del registro del sistema: eigenestado |1⟩
qc.x(n_ancilla)  # |0⟩ → |1⟩
qc.barrier()

# Aplicar U^(2^j) controladas — U = T gate (T|1⟩ = e^(iπ/4)|1⟩)
for j in range(n_ancilla):
    reps = 2**j
    for _ in range(reps):
        qc.cp(np.pi/4, j, n_ancilla)  # Controlled-T

qc.barrier()

In [ ]:
# QFT inversa sobre el registro ancilla
def iqft(qc, qubits):
    n = len(qubits)
    for i in range(n//2):
        qc.swap(qubits[i], qubits[n-i-1])
    for i in range(n-1, -1, -1):
        qc.h(qubits[i])
        for j in range(i-1, -1, -1):
            angle = -np.pi / 2**(i-j)
            qc.cp(angle, qubits[i], qubits[j])

iqft(qc, list(range(n_ancilla)))
qc.measure(range(n_ancilla), range(n_ancilla))
print(qc.draw())

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

sim = AerSimulator()
qc_t = transpile(qc, sim)
job = sim.run(qc_t, shots=1000)
conteos = job.result().get_counts()

print('Conteos:', conteos)
# El estado más probable codifica φ
best = max(conteos, key=conteos.get)
phi_estimado = int(best, 2) / 2**n_ancilla
print(f'Fase estimada: {phi_estimado:.4f}')
print(f'Fase real:     {1/8:.4f} = 1/8')

## Precisión vs número de ancillas

In [ ]:
phi_real = 1/8
phi_str = '1/8'

for n_a in [2, 3, 4, 5]:
    error = abs(phi_real - round(phi_real * 2**n_a) / 2**n_a)
    print(f'n_ancilla={n_a}: error máx = {1/2**n_a:.4f}, error en φ={phi_str}: {error:.6f}')

## Fase arbitraria

Cuando φ no es exactamente representable en n_ancilla bits, la salida tiene distribución probabilística alrededor del valor más cercano:

In [ ]:
# φ = 1/3 — no representable exactamente en 3 bits
phi_aprox = 1/3
print(f'φ = 1/3 en 3 bits: el más cercano es {round(phi_aprox*8)/8:.4f}')
print(f'Error: {abs(phi_aprox - round(phi_aprox*8)/8):.4f}')

## Ejercicio

1. Modifica el circuito para estimar la fase del gate S (φ = 1/4).
2. ¿Cuántas ancillas necesitas para precisión < 0.01?
3. ¿Qué pasa si el sistema no está en un eigenestado de U?

In [ ]:
# Tu solución — S gate tiene φ = 1/4
print('Para φ=1/4, n_ancilla mínimo tal que error < 0.01:')
for n in range(1, 10):
    if 1/2**n < 0.01:
        print(f'  n_ancilla = {n} (error máx = {1/2**n:.4f})')
        break

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/07_phase_estimation_guiada.ipynb`
- **Algoritmo de Shor:** `07_entrelazamiento.ipynb` → `12_algoritmo_shor.ipynb`
- **Módulo teórico:** `Tutorial/05_algoritmos/README.md`